# Building a RAG System with LangChain and FAISS

## Introduction to RAG (Retrieval-Augmented Generation)

RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

---

## FAISS

https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

### Key advantages:

1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU Accelaration
4. Can Handle millions of Vectors

How it Works
    - Indexes Vectors for fast neatest Neighbour Search
    - Returns most Similar Vectors based on the distance metrics

In [1]:
## Importing the Libraries
import os
from dotenv import load_dotenv
import numpy as np
from typing import List,Dict,Any,Optional
import warnings
warnings.filterwarnings('ignore')

In [3]:
## Langchain Core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

## Loading the Enviroment Variables
load_dotenv()

True

## Data Ingestion and Processing

In [5]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

### Perform the Text Splitting

In [6]:
## Perform the Text Splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "]
)

## Split the Documents into Chunks
chunks = text_splitter.split_documents(sample_documents)
print(chunks)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognit

In [8]:
print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
for i in range(0,len(chunks)):
    print(f"Content: {chunks[i].page_content}")
    print(f"MetaData: {chunks[i].metadata}")

Created 4 chunks from 4 documents
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
MetaData: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
MetaData: {'source': 'ML Basics', 'page': 1, 'topic': 'ML'}
Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
MetaData: {'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}
Content: Natural

In [9]:
## Load the Embedding Models
import os
load_dotenv()
os.environ["HUGGINGFACE_API_KEY"] = os.getenv("HUGGINGFACE_API_KEY")

In [10]:
## Initialize the HuggingFace Embeddings
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

## Example: Create an EMbedding for a Single Text
simple_text = "What is Machine Learning?"
sample_embeddings = embeddings.embed_query(simple_text)
print(sample_embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6432.12it/s]


[-0.019954543560743332, 0.009877996519207954, 0.010249625891447067, 0.029553672298789024, 0.027186455205082893, -0.01929653249680996, -0.024129599332809448, -0.03773573040962219, -0.04105418175458908, -0.001474996330216527, -0.07607676088809967, 0.036872051656246185, 0.05372382700443268, -0.07053525000810623, -0.0860934630036354, 0.021006811410188675, -0.04226336628198624, 0.028448954224586487, -0.03837539255619049, -0.06591922044754028, 0.013949659653007984, 0.01227542944252491, -0.06821969151496887, 0.02862505242228508, 0.014761139638721943, 0.03223147988319397, 0.008694248273968697, 0.013062095269560814, -0.015726717188954353, 0.018896566703915596, 0.020023835822939873, -0.039297088980674744, 0.015647148713469505, 0.03339800611138344, -0.07339101284742355, 0.059357818216085434, -0.012385796755552292, 0.024584665894508362, 0.020107105374336243, -0.020254511386156082, -0.05024771764874458, -0.09044116735458374, -0.015374377369880676, -0.003612410742789507, 0.1299968957901001, 0.116227

In [ ]:
text = ["AI","Machine Learning","Deep Learning","Neural Networks"]
batch_embeddings = embeddings.embed_documents(text)
print(batch_embeddings)

print(batch_embeddings[0]) ## Embeddings for the AI

[[-0.03653925284743309, -0.015164326876401901, 0.01643250323832035, 0.010568861849606037, 0.006010537035763264, -0.01847331039607525, 0.08546523749828339, 0.02096843533217907, 0.027815362438559532, 0.012431688606739044, -0.029376521706581116, -0.03113526478409767, 0.034912560135126114, -0.018150823190808296, -0.06498483568429947, 0.0516824908554554, -0.019606225192546844, -0.015734195709228516, -0.13371677696704865, -0.09645990282297134, -0.025471771135926247, -0.0014895489439368248, -0.00634934613481164, -0.02582067996263504, -0.02737179957330227, 0.12268996238708496, -0.007792518474161625, -0.038522738963365555, 0.014383512549102306, -0.09218425303697586, 0.00869576446712017, 0.002613399177789688, 0.09103471040725708, -0.030313605442643166, -0.09604637324810028, 0.022289128974080086, -0.0902431309223175, -0.03294737637042999, 0.07158337533473969, -0.008893121033906937, -0.025708982720971107, -0.07913965731859207, 0.014530361630022526, -0.07420428842306137, 0.08045009523630142, 0.0780

In [13]:
## Comparing the Embeddings using Cosine Similarity

def compare_embeddings(text1: str,text2: str):
    """ Compare the Semantic Similarity of 2 Texts using the Embeddings"""
    
    emb1 = np.array(embeddings.embed_query(text=text1))
    emb2 = np.array(embeddings.embed_query(text=text2))
    
    ## Calculate the SImilarity Score
    similarity = (np.dot(emb1,emb2))/(np.linalg.norm(emb1)*np.linalg.norm(emb2))
    return similarity

In [14]:
## Testing the Semantic Similarity
print(f"'AI' vs 'ML' : {compare_embeddings('AI','ML')}")

'AI' vs 'ML' : 0.3438046800351837


### Create a FAISS Vector Store

In [15]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector Store created with {vector_store.index.ntotal} vectors")

Vector Store created with 4 vectors


In [16]:
vector_store

In [17]:
## Save the vector Store for later use
vector_store.save_local("faiss_index")

In [18]:
## Load the Vector Store
loaded_vector_store = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
print(f"Vector Store created with {vector_store.index.ntotal} vectors")

Vector Store created with 4 vectors


In [19]:
## Similarity Search
query = "What is Machine Learning"
results = vector_store.similarity_search(query, k = 3)
print(results)

[Document(id='35e2517a-dec3-4d91-971a-6a12fed9b692', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='9336f3db-4e0d-4403-9cab-595939a3fdb2', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='d96b6721-c3e2-4dce-87ec-811183b5fb86', metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        T

In [20]:
print(f"Query: {query}")
print("Top 3 Similar Chunks")
for i,doc in enumerate(results):
    print(f"\n{i+1}, Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content[:200]} .....")

Query: What is Machine Learning
Top 3 Similar Chunks

1, Source: ML Basics
Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised .....

2, Source: Deep Learning
Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning  .....

3, Source: AI Introduction
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into na .....


In [21]:
## Similarity Search with Score
result_with_score = vector_store.similarity_search_with_score(query,k=3)
print("\n\n Similarty Search with Scores")
for doc,score in result_with_score:
    print(f"\n Score: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"COntent Preview: {doc.page_content[:100]}..")



 Similarty Search with Scores

 Score: 0.471
Source: ML Basics
COntent Preview: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being..

 Score: 0.954
Source: Deep Learning
COntent Preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m..

 Score: 1.058
Source: AI Introduction
COntent Preview: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These syst..


In [22]:
chunks

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recogn

In [23]:
### Search with metadata filtering
filter_dic = {"topic":"ML"}
filtered_results= vector_store.similarity_search(
    query,
    k=3,
    filter = filter_dic
)
print(filtered_results)

[Document(id='35e2517a-dec3-4d91-971a-6a12fed9b692', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


In [24]:
len(filtered_results)

1